# Counter-Propagating Fields

Many atomic physics experiments use fields that propagate in opposite directions through
the medium. A common example is Rydberg EIT, where a near-infrared probe and a blue
coupling field enter the vapour cell from opposite ends to reduce (or eliminate) the
Doppler broadening of the two-photon resonance.

## Why a counter-propagating geometry changes the Doppler shift

An atom moving with velocity $v$ along the propagation axis sees each field at a
Doppler-shifted frequency. For a field with wave-vector $\mathbf{k}$:

- **Co-propagating** ($\mathbf{k} \parallel \hat{z}$): the atom sees the field
  blue-shifted by $kv$, so its effective one-photon detuning shifts by $+kv$.
- **Counter-propagating** ($\mathbf{k} \antiparallel \hat{z}$): the shift is
  $-kv$.

In a two-photon (Ξ or Λ) scheme, the _two-photon_ detuning is the sum of the
individual detunings. When both fields propagate in the same direction, the
two-photon detuning picks up $+kv + kv = 2kv$, and the resonance is
Doppler-broadened. When the fields counter-propagate with similar wavelengths, the
shifts partially or fully cancel, giving a narrower (Doppler-reduced or
Doppler-free) two-photon resonance.

## How MaxwellBloch handles this

MaxwellBloch propagates all fields in the $+z$ direction (it solves an
initial-value problem marching from $z_\mathrm{min}$ to $z_\mathrm{max}$). A
genuinely counter-propagating field would require boundary data at $z_\mathrm{max}$
and backward integration, turning the problem into a boundary-value problem.

However, when the counter-propagating field **does not deplete significantly**
across the cell—a valid approximation for many coupling fields in EIT
experiments—the forward-propagating solver is equivalent to the true
counter-propagating geometry. The only physical difference is the **sign of the
Doppler shift** each velocity class contributes to that field. MaxwellBloch exposes
this sign via the `counter_propagating` field attribute.

> **Note on phrasing.** This is *not* an 'undepleted-coupling approximation' in the
> sense of neglecting absorption. The solver computes absorption honestly for every
> velocity class. The approximation is purely geometric: it exploits the spatial
> symmetry that holds when the counter-propagating field's intensity profile is
> essentially flat across the cell.

## Basic usage: `counter_propagating`

Set `"counter_propagating": true` on any field in the JSON configuration. This
tells MaxwellBloch to apply the Doppler shift with the opposite sign for that field,
while leaving the forward-propagation geometry unchanged.

In [ ]:
mb_solve_json = """
{
  "atom": {
    "num_states": 3,
    "decays": [{"channels": [[0, 1], [1, 2]], "rate": 1.0}],
    "fields": [
      {
        "label": "probe",
        "coupled_levels": [[0, 1]],
        "rabi_freq": 1.0e-3,
        "rabi_freq_t_func": "gaussian",
        "rabi_freq_t_args": {"ampl": 1.0, "centre": 0.0, "fwhm": 1.0}
      },
      {
        "label": "coupling",
        "coupled_levels": [[1, 2]],
        "rabi_freq": 5.0,
        "rabi_freq_t_func": "ramp_onoff",
        "rabi_freq_t_args": {"ampl": 1.0, "fwhm": 0.2, "on": -2.0, "off": 8.0},
        "counter_propagating": true
      }
    ]
  },
  "t_min": -2.0,
  "t_max": 8.0,
  "t_steps": 120,
  "z_min": 0.0,
  "z_max": 1.0,
  "z_steps": 20,
  "z_steps_inner": 2,
  "interaction_strengths": [1.0, 1.0],
  "velocity_classes": {
    "thermal_width": 5.0,
    "thermal_delta_min": -15.0,
    "thermal_delta_max": 15.0,
    "thermal_delta_steps": 10
  },
  "savefile": "counter-propagating-eit"
}
"""

In [ ]:
from maxwellbloch import mb_solve

mbs = mb_solve.MBSolve().from_json_str(mb_solve_json)
print(f"Coupling field counter_propagating: {mbs.atom.fields[1].counter_propagating}")
print(f"Coupling factor_doppler_shift:      {mbs.atom.fields[1].factor_doppler_shift}")

In [ ]:
Omegas_zt, states_zt = mbs.mbsolve()

### Depletion check

After solving, MaxwellBloch automatically checks whether the counter-propagating
field depleted significantly across the cell. The check computes

$$
\text{depletion} = 1 - \frac{\max_t |\Omega(z_\mathrm{max}, t)|}{\max_t |\Omega(z_\mathrm{min}, t)|}
$$

- **< 1 %** — check passes silently. The forward-propagation result is reliable.
- **1 – 10 %** — a `UserWarning` is emitted. Results may still be usable but
  caution is advised.
- **> 10 %** — a `CounterPropagatingDepletionError` is raised. The approximation
  has broken down.

You can suppress the check with `check_counter_prop_depletion=False` if you know
the depletion is acceptable for your purposes.

In [ ]:
import numpy as np

coupling_idx = 1  # second field
Omega_coupling = Omegas_zt[coupling_idx]
peak_in  = np.max(np.abs(Omega_coupling[0]))
peak_out = np.max(np.abs(Omega_coupling[-1]))
depletion = 1.0 - peak_out / peak_in
print(f"Coupling field depletion: {depletion:.2%}")

## Advanced: `factor_doppler_shift`

For schemes where the two counter-propagating fields have **different wavelengths**
(e.g. a three-photon Doppler-free ladder where $k_1 \neq k_2$), the Doppler
cancellation is only partial. Use `factor_doppler_shift` directly to set the
fractional weight:

```json
{
  "coupled_levels": [[2, 3]],
  "rabi_freq": 1.5,
  "factor_doppler_shift": -0.62
}
```

The sign determines the direction; the magnitude scales the Doppler shift relative
to the thermal width unit. `factor_doppler_shift = 1.0` (the default) gives
standard co-propagating behaviour; `-1.0` is equivalent to
`counter_propagating = true`.

> **Note on `detuning_positive`.** The `detuning_positive` flag on a field is a
> **separate and orthogonal** concept. It controls the sign convention for the RWA
> Hamiltonian based on the level ordering (used when the second coupled level sits
> below the first in energy, e.g. the lower leg of a Λ system). It has nothing to
> do with the laser k-vector direction. Both flags can be set independently.

## Pulse-area check

As a sanity check, verify that the probe pulse area at the input is correct.

In [ ]:
probe_area_in = np.trapezoid(np.abs(Omegas_zt[0, 0, :]), mbs.tlist) / np.pi
print(f"Probe area at z=z_min: {probe_area_in:.4f} π")

## References

1. Mohapatra, Jackson, Adams — *Coherent Optical Detection of Highly Excited
   Rydberg States Using Electromagnetically Induced Transparency*,
   PRL **98**, 113003 (2007). Example of the counter-propagating Rydberg EIT
   geometry this feature supports.
2. Adams, Pritchard, Shaffer — *Rydberg atom physics*,
   J. Phys. B **53**, 012002 (2020). Broader review including Doppler-free
   multi-photon spectroscopy.